# DATASCAPE 2026 — Preprocessing Pipeline

Notebook Google Colab mandiri untuk membentuk data training yang bersih tanpa mengubah file sumber.

**Kebijakan V1**

- verifikasi seluruh file dan keluarkan file yang tidak dapat dibaca;
- terapkan orientasi EXIF dan konversi semua input menjadi RGB saat runtime;
- keluarkan semua kelompok exact-pixel duplicate yang memiliki label berbeda;
- pertahankan satu file kanonik untuk exact duplicate dengan label sama;
- pertahankan seluruh gambar unik kelas `Lainnya`;
- near-duplicate hanya dipakai sebagai grup split pada V1, bukan langsung dihapus;
- gunakan letterbox/padding untuk input model; koreksi brightness, CLAHE, dan crop agresif belum diaktifkan;
- label asli tidak diubah otomatis.

Jalankan **Runtime → Run all**. Dataset mentah tetap berada di lokasi Kaggle/Colab dan tidak dimodifikasi.


In [ ]:
# Kaggle API bootstrap (configured for this project)
import os
from pathlib import Path

KAGGLE_API_TOKEN = 'KGAT_c2a604d54464b1bc7271fb95428085f2'
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

kaggle_directory = Path.home() / '.kaggle'
kaggle_directory.mkdir(parents=True, exist_ok=True)
kaggle_access_token = kaggle_directory / 'access_token'
kaggle_access_token.write_text(KAGGLE_API_TOKEN + '\n')
kaggle_access_token.chmod(0o600)
print('Kaggle API token configured:', kaggle_access_token)


## 1. Kepatuhan dan batas penggunaan model

Q&A panitia per 2 Oktober 2026 memperbolehkan model pretrained, termasuk image encoder CLIP/VLM tanpa bagian teks, jika di-fine-tune menggunakan data train resmi. Model eksternal tidak boleh digunakan langsung untuk inferensi atau relabeling test set.

Notebook ini tetap memakai model `weights=None` untuk audit label opsional. Ini adalah pilihan desain konservatif agar sinyal audit sepenuhnya berasal dari training resmi, bukan kewajiban aturan lomba. Prediksi out-of-fold hanya dibuat atas training set.

Model out-of-fold dapat menghasilkan **kandidat** label bermasalah, tetapi tidak membuktikan label benar. Gambar pixel-identik dengan label berbeda dikeluarkan karena model tidak mungkin membedakannya dari visual yang sama.


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow',
    'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn',
    'tqdm': 'tqdm', 'cv2': 'opencv-python-headless',
    'torch': 'torch', 'torchvision': 'torchvision',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
import hashlib
import io
import json
import math
import os
import random
import shutil
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm
from IPython.display import display, Markdown

SEED = 42
CLASSES = ('Aman', 'Rusak', 'Lainnya')
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
IN_COLAB = 'google.colab' in sys.modules
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Colab:', IN_COLAB, '| Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())


In [ ]:
# Konfigurasi utama — default dapat langsung dijalankan.
COMPETITION = 'datascape-2026-data-mining-competition'
DATA_SOURCE = 'auto'       # auto | folder | kaggle
LOCAL_DATA_PATH = ''       # isi jika DATA_SOURCE='folder'
OUTPUT_DIR = Path('/content/datascape_preprocessing') if IN_COLAB else Path.cwd() / 'artifacts' / 'preprocessing_colab'
EXTRACT_DIR = Path('/content/datascape_dataset') if IN_COLAB else Path.cwd() / 'data' / 'datascape_colab'
WORKERS = 4
IMAGE_SIZE = 320
FOLDS = 5

# Near-duplicate V1: threshold sengaja ketat agar salah-grouping rendah.
NEAR_PHASH_MAX = 6
NEAR_DHASH_MAX = 8
NEAR_COLOR_COSINE_MIN = 0.98
NEAR_ASPECT_RELATIVE_MAX = 0.05

# Koreksi foto dan relabeling tidak diterapkan.
ENABLE_PHOTOMETRIC_AUGMENTATION = False

# Gunakan Colab Secret atau environment KAGGLE_API_TOKEN bila dataset perlu diunduh.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def get_kaggle_token():
    token = os.environ.get('KAGGLE_API_TOKEN', '').strip()
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = (userdata.get('KAGGLE_API_TOKEN') or token).strip()
        except Exception:
            pass
    if not token:
        raise RuntimeError('Set KAGGLE_API_TOKEN di Colab Secrets atau environment.')
    return token


def safe_extract(zip_path: Path, destination: Path):
    destination.mkdir(parents=True, exist_ok=True)
    resolved = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(resolved):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)


def find_dataset_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start]
    if start.is_dir():
        candidates.extend(p.parent for p in start.rglob('sample_submission.csv'))
    for root in candidates:
        if all((root / 'Train_Set' / label).is_dir() for label in CLASSES) and (root / 'Test_Set').is_dir():
            return root
    raise FileNotFoundError(f'Dataset root tidak ditemukan di {start}')


def resolve_dataset():
    if DATA_SOURCE in {'auto', 'folder'} and LOCAL_DATA_PATH:
        path = Path(LOCAL_DATA_PATH).expanduser()
        if path.exists():
            return find_dataset_root(path)
        if DATA_SOURCE == 'folder':
            raise FileNotFoundError(path)
    local = Path.cwd() / COMPETITION
    if DATA_SOURCE == 'auto' and local.exists():
        return find_dataset_root(local)

    if importlib.util.find_spec('kagglehub') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    from kagglehub.config import set_kaggle_api_token
    set_kaggle_api_token(get_kaggle_token())
    downloaded = Path(kagglehub.competition_download(COMPETITION))
    if downloaded.is_file() and downloaded.suffix.lower() == '.zip':
        safe_extract(downloaded, EXTRACT_DIR)
        return find_dataset_root(EXTRACT_DIR)
    return find_dataset_root(downloaded)


DATA_ROOT = resolve_dataset()
print('Dataset root:', DATA_ROOT)
for label in CLASSES:
    count = sum(p.is_file() and p.suffix.lower() in EXTENSIONS for p in (DATA_ROOT / 'Train_Set' / label).rglob('*'))
    print(f'{label}: {count:,}')


## 2. Audit file, kanal warna, dimensi, dan hash

`pixel_sha256` dihitung setelah orientasi EXIF dan konversi RGB. Dua file dengan hash ini sama memberikan tensor visual yang identik walaupun ekstensi atau metadata filenya berbeda.

Statistik warna dihitung pada thumbnail 128×128 untuk efisiensi. Nilai ini hanya metadata audit, bukan koreksi gambar.


In [ ]:
def load_rgb_image(source) -> Image.Image:
    with Image.open(source) as image:
        if image.mode == 'P':
            image = image.convert('RGBA')
        image = ImageOps.exif_transpose(image)
        if image.mode in {'RGBA', 'LA'} or 'transparency' in image.info:
            rgba = image.convert('RGBA')
            result = Image.new('RGB', rgba.size, (128, 128, 128))
            result.paste(rgba, mask=rgba.getchannel('A'))
            return result
        return image.convert('RGB')


def hash_bits_to_hex(bits):
    return np.packbits(np.asarray(bits, dtype=np.uint8)).tobytes().hex()


def perceptual_hashes(rgb_small):
    gray32 = cv2.resize(cv2.cvtColor(rgb_small, cv2.COLOR_RGB2GRAY), (32, 32), interpolation=cv2.INTER_AREA)
    dct = cv2.dct(gray32.astype(np.float32))[:8, :8]
    phash = dct > np.median(dct.flat[1:])
    gray9x8 = cv2.resize(gray32, (9, 8), interpolation=cv2.INTER_AREA)
    dhash = gray9x8[:, 1:] > gray9x8[:, :-1]
    return hash_bits_to_hex(phash.ravel()), hash_bits_to_hex(dhash.ravel())


def inspect_image(path: Path):
    row = {
        'filepath': str(path.relative_to(DATA_ROOT)),
        'filename': path.name,
        'label': path.relative_to(DATA_ROOT / 'Train_Set').parts[0],
    }
    try:
        content = path.read_bytes()
        row.update(file_bytes=len(content), file_sha256=hashlib.sha256(content).hexdigest())
        with Image.open(io.BytesIO(content)) as raw:
            raw.verify()
        with Image.open(io.BytesIO(content)) as raw:
            row.update(format=raw.format, raw_mode=raw.mode, exif_orientation=int(raw.getexif().get(274, 1)))
        rgb = load_rgb_image(io.BytesIO(content))
        width, height = rgb.size
        pixel_digest = hashlib.sha256(f'{width}x{height}:RGB:'.encode() + rgb.tobytes()).hexdigest()
        small = np.asarray(rgb.resize((128, 128), Image.Resampling.LANCZOS), dtype=np.uint8)
        values = small.astype(np.float32) / 255.0
        luminance = 0.2126 * values[..., 0] + 0.7152 * values[..., 1] + 0.0722 * values[..., 2]
        phash, dhash = perceptual_hashes(small)
        histogram = cv2.calcHist([small], [0, 1, 2], None, [4, 4, 4], [0, 256, 0, 256, 0, 256]).ravel()
        histogram = histogram / max(float(histogram.sum()), 1.0)
        means = values.mean(axis=(0, 1))
        squares = np.square(values).mean(axis=(0, 1))
        row.update(
            width=width, height=height, aspect_ratio=width / height,
            pixel_sha256=pixel_digest, phash=phash, dhash=dhash,
            color_histogram=';'.join(f'{value:.8f}' for value in histogram),
            luminance_mean=float(luminance.mean()), luminance_std=float(luminance.std()),
            luminance_p01=float(np.percentile(luminance, 1)),
            luminance_p99=float(np.percentile(luminance, 99)),
            red_mean=float(means[0]), green_mean=float(means[1]), blue_mean=float(means[2]),
            red_sq_mean=float(squares[0]), green_sq_mean=float(squares[1]), blue_sq_mean=float(squares[2]),
            error='',
        )
    except Exception as exc:
        row['error'] = f'{type(exc).__name__}: {exc}'
    return row


train_paths = sorted(
    p for label in CLASSES for p in (DATA_ROOT / 'Train_Set' / label).rglob('*')
    if p.is_file() and p.suffix.lower() in EXTENSIONS
)
metadata_path = OUTPUT_DIR / 'image_metadata.csv'
if metadata_path.exists():
    metadata = pd.read_csv(metadata_path)
    if set(metadata.filepath) != {str(p.relative_to(DATA_ROOT)) for p in train_paths}:
        print('Cache metadata tidak cocok; audit dijalankan ulang.')
        metadata = None
else:
    metadata = None

if metadata is None:
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        metadata = pd.DataFrame(list(tqdm(pool.map(inspect_image, train_paths), total=len(train_paths), desc='Audit images')))
    metadata.to_csv(metadata_path, index=False)

print('Total:', len(metadata), '| readable:', metadata.error.fillna('').eq('').sum(), '| unreadable:', metadata.error.fillna('').ne('').sum())
display(metadata.loc[metadata.error.fillna('').ne(''), ['filepath', 'label', 'error']])
display(pd.crosstab(metadata.label, metadata.raw_mode, dropna=False).reindex(CLASSES))


## 3. Exact deduplication dan manifest V1

Pemilihan file kanonik bersifat deterministik. Karena decoded RGB dalam satu grup identik, prioritas diberikan kepada ekstensi yang cocok dengan format internal, mode RGB biasa, ukuran file lebih kecil, lalu path alfabetis.


In [ ]:
EXPECTED_FORMAT = {'.jpg': 'JPEG', '.jpeg': 'JPEG', '.png': 'PNG', '.webp': 'WEBP', '.bmp': 'BMP'}
frame = metadata.copy()
frame['decision'] = ''
frame['decision_reason'] = ''
error_text = frame.error.fillna('')
unreadable = error_text.ne('')
frame.loc[unreadable, ['decision', 'decision_reason']] = ['exclude', 'unreadable_image']

readable = frame[~unreadable].copy()
group_stats = readable.groupby('pixel_sha256').agg(
    group_size=('filepath', 'size'), label_count=('label', 'nunique')
)
conflict_hashes = set(group_stats.index[group_stats.label_count.gt(1)])
conflicting = frame.pixel_sha256.isin(conflict_hashes) & ~unreadable
frame.loc[conflicting, ['decision', 'decision_reason']] = ['exclude', 'exact_cross_label_conflict']

eligible = frame[frame.decision.eq('')].copy()
eligible['extension'] = eligible.filename.map(lambda value: Path(value).suffix.lower())
eligible['format_matches_extension'] = eligible.apply(
    lambda row: EXPECTED_FORMAT.get(row.extension) == row['format'], axis=1
)
eligible['plain_rgb'] = eligible.raw_mode.eq('RGB')
eligible = eligible.sort_values(
    ['format_matches_extension', 'plain_rgb', 'file_bytes', 'filepath'],
    ascending=[False, False, True, True], kind='stable'
)
keep_paths = set(eligible.drop_duplicates('pixel_sha256', keep='first').filepath)
same_label_duplicate = frame.decision.eq('') & ~frame.filepath.isin(keep_paths)
frame.loc[same_label_duplicate, ['decision', 'decision_reason']] = ['exclude', 'exact_same_label_duplicate']
frame.loc[frame.decision.eq(''), ['decision', 'decision_reason']] = ['include', 'canonical_or_unique']

manifest = frame[frame.decision.eq('include')].copy().reset_index(drop=True)
exclusions = frame[frame.decision.eq('exclude')].copy().reset_index(drop=True)
manifest['source_path'] = manifest.filepath.map(lambda value: str((DATA_ROOT / value).resolve()))

assert not manifest.pixel_sha256.duplicated().any()
assert set(manifest.label) == set(CLASSES)
manifest.to_csv(OUTPUT_DIR / 'training_manifest_exact.csv', index=False)
exclusions.to_csv(OUTPUT_DIR / 'excluded_images.csv', index=False)

print('Manifest exact-deduplicated:')
display(manifest.label.value_counts().reindex(CLASSES).rename('images').to_frame())
print('Exclusions:')
display(pd.crosstab(exclusions.decision_reason, exclusions.label).reindex(columns=CLASSES, fill_value=0))


## 4. Kandidat near-duplicate dan grup anti-leakage

Tahap ini hanya memakai pHash, dHash, histogram warna, dimensi, dan pixel resmi. Threshold dibuat ketat. Near-duplicate tetap disimpan; `split_group` memastikan kandidat yang berhubungan tidak masuk fold berbeda.

Grouping heuristik dapat menggabungkan foto yang sangat mirip tetapi bukan salinan. Karena itu V1 tidak menghapusnya.


In [ ]:
def hex_to_bits(value):
    return np.unpackbits(np.frombuffer(bytes.fromhex(value), dtype=np.uint8))


def histogram_array(value):
    return np.fromstring(value, sep=';', dtype=np.float32)


class UnionFind:
    def __init__(self, size):
        self.parent = list(range(size))
        self.rank = [0] * size
    def find(self, value):
        while self.parent[value] != value:
            self.parent[value] = self.parent[self.parent[value]]
            value = self.parent[value]
        return value
    def union(self, left, right):
        left, right = self.find(left), self.find(right)
        if left == right:
            return
        if self.rank[left] < self.rank[right]:
            left, right = right, left
        self.parent[right] = left
        if self.rank[left] == self.rank[right]:
            self.rank[left] += 1


phash_bits = np.stack([hex_to_bits(value) for value in manifest.phash])
dhash_bits = np.stack([hex_to_bits(value) for value in manifest.dhash])
histograms = np.stack([histogram_array(value) for value in manifest.color_histogram])
nearest = NearestNeighbors(n_neighbors=min(12, len(manifest)), metric='hamming', algorithm='brute', n_jobs=-1)
nearest.fit(phash_bits)
phash_distances, phash_indices = nearest.kneighbors(phash_bits)

pairs = []
seen = set()
for left in tqdm(range(len(manifest)), desc='Verify near duplicates'):
    for distance, right in zip(phash_distances[left, 1:], phash_indices[left, 1:]):
        right = int(right)
        key = (min(left, right), max(left, right))
        if key in seen:
            continue
        seen.add(key)
        phash_hamming = int(round(float(distance) * 64))
        if phash_hamming > NEAR_PHASH_MAX:
            continue
        dhash_hamming = int(np.count_nonzero(dhash_bits[left] != dhash_bits[right]))
        if dhash_hamming > NEAR_DHASH_MAX:
            continue
        aspect_left, aspect_right = manifest.iloc[left].aspect_ratio, manifest.iloc[right].aspect_ratio
        aspect_relative = abs(aspect_left - aspect_right) / max(aspect_left, aspect_right)
        if aspect_relative > NEAR_ASPECT_RELATIVE_MAX:
            continue
        color_cosine = float(histograms[left] @ histograms[right] / (
            max(np.linalg.norm(histograms[left]), 1e-12) * max(np.linalg.norm(histograms[right]), 1e-12)
        ))
        if color_cosine < NEAR_COLOR_COSINE_MIN:
            continue
        pairs.append({
            'filepath_a': manifest.iloc[left].filepath,
            'label_a': manifest.iloc[left].label,
            'filepath_b': manifest.iloc[right].filepath,
            'label_b': manifest.iloc[right].label,
            'phash_hamming': phash_hamming,
            'dhash_hamming': dhash_hamming,
            'color_cosine': color_cosine,
            'aspect_relative_difference': aspect_relative,
            'cross_label': manifest.iloc[left].label != manifest.iloc[right].label,
            'left_index': left, 'right_index': right,
        })

near_pairs = pd.DataFrame(pairs)
if len(near_pairs):
    near_pairs.drop(columns=['left_index', 'right_index']).to_csv(OUTPUT_DIR / 'near_duplicate_candidates.csv', index=False)
else:
    pd.DataFrame(columns=['filepath_a','label_a','filepath_b','label_b']).to_csv(OUTPUT_DIR / 'near_duplicate_candidates.csv', index=False)

union = UnionFind(len(manifest))
for row in pairs:
    union.union(row['left_index'], row['right_index'])
roots = [union.find(index) for index in range(len(manifest))]
root_to_group = {root: f'near_{number:05d}' for number, root in enumerate(sorted(set(roots)))}
manifest['split_group'] = [root_to_group[root] for root in roots]
print('Near-duplicate candidate pairs:', len(near_pairs))
print('Cross-label candidate pairs:', int(near_pairs.cross_label.sum()) if len(near_pairs) else 0)
print('Split groups:', manifest.split_group.nunique())


## 5. Grouped 5-fold split

Satu `split_group` hanya boleh berada dalam satu fold. Fold ini dipakai untuk evaluasi, tuning preprocessing, dan prediksi out-of-fold. Test tidak digunakan untuk fitting statistik atau memilih preprocessing.


In [ ]:
manifest['fold'] = -1
splitter = StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
for fold, (_, valid_indices) in enumerate(splitter.split(manifest, manifest.label, groups=manifest.split_group)):
    manifest.loc[valid_indices, 'fold'] = fold

assert manifest.fold.ge(0).all()
assert manifest.groupby('split_group').fold.nunique().max() == 1
for fold in range(FOLDS):
    assert set(manifest.loc[manifest.fold.eq(fold), 'label']) == set(CLASSES)

manifest.to_csv(OUTPUT_DIR / 'training_manifest.csv', index=False)
fold_table = pd.crosstab(manifest.fold, manifest.label).reindex(columns=CLASSES)
display(fold_table)
print('Total:', len(manifest), '| groups:', manifest.split_group.nunique())


## 6. Statistik normalisasi RGB

Mean dan standard deviation dihitung hanya dari gambar training yang masuk manifest. Statistik memakai thumbnail berukuran sama sehingga setiap gambar mendapat bobot yang setara.


In [ ]:
channel_mean = np.array([
    manifest.red_mean.mean(), manifest.green_mean.mean(), manifest.blue_mean.mean()
])
channel_second = np.array([
    manifest.red_sq_mean.mean(), manifest.green_sq_mean.mean(), manifest.blue_sq_mean.mean()
])
channel_std = np.sqrt(np.maximum(channel_second - channel_mean ** 2, 1e-12))
print('RGB mean:', channel_mean.round(6).tolist())
print('RGB std :', channel_std.round(6).tolist())
(OUTPUT_DIR / 'normalization.json').write_text(json.dumps({
    'mean': channel_mean.tolist(), 'std': channel_std.tolist(), 'source': 'training_manifest thumbnails only'
}, indent=2))


## 7. Transform baseline: RGB + letterbox

Baseline mempertahankan seluruh gambar dan aspect ratio. Padding memakai mean RGB training. Brightness/contrast correction tidak diterapkan. Augmentasi fotometrik dapat dinyalakan setelah baseline dinilai.


In [ ]:
class ResizeWithPadding:
    def __init__(self, size, fill):
        self.size = int(size)
        self.fill = tuple(int(round(value * 255)) for value in fill)
    def __call__(self, image):
        return ImageOps.pad(
            image, (self.size, self.size), method=Image.Resampling.BILINEAR,
            color=self.fill, centering=(0.5, 0.5),
        )


def make_transform(training=False):
    operations = [ResizeWithPadding(IMAGE_SIZE, channel_mean)]
    if training:
        operations.extend([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(5, fill=tuple(int(round(value * 255)) for value in channel_mean)),
        ])
        if ENABLE_PHOTOMETRIC_AUGMENTATION:
            operations.append(transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02))
    operations.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=channel_mean.tolist(), std=channel_std.tolist()),
    ])
    return transforms.Compose(operations)


class ManifestDataset(Dataset):
    def __init__(self, frame, training=False, return_path=False):
        self.frame = frame.reset_index(drop=True)
        self.transform = make_transform(training)
        self.return_path = return_path
        self.class_to_index = {label: index for index, label in enumerate(CLASSES)}
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = load_rgb_image(row.source_path)
        tensor = self.transform(image)
        target = self.class_to_index[row.label]
        if self.return_path:
            return tensor, target, row.filepath
        return tensor, target


def make_balanced_sampler(frame, strength=0.5):
    # strength=0 -> natural distribution; 0.5 -> sqrt inverse frequency;
    # strength=1 -> fully class-balanced sampling.
    counts = frame.label.value_counts()
    weights = frame.label.map(lambda label: float(counts[label]) ** (-strength)).to_numpy()
    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), len(frame), replacement=True)

sample_dataset = ManifestDataset(manifest.head(8), training=False)
images, labels = zip(*(sample_dataset[index] for index in range(len(sample_dataset))))
print('Tensor shape:', images[0].shape, '| RGB normalization ready')


## 8. Global + local tiles untuk paket kecil

Cropping tidak menciptakan detail baru. Ia membuang latar agar pixel objek menggunakan area input model yang lebih besar. Untuk mencegah hilangnya konteks, selalu pertahankan satu global view.

Fungsi berikut menghasilkan global view dan empat tile ber-overlap untuk gambar yang cukup besar. Gambar kecil hanya memakai global view agar crop 100×100 tidak diperbesar secara berlebihan.


In [ ]:
def global_and_local_views(image, size=IMAGE_SIZE, overlap=0.15, minimum_tile_source=160):
    image = image.convert('RGB')
    global_view = ResizeWithPadding(size, channel_mean)(image)
    width, height = image.size
    if min(width, height) < minimum_tile_source * 2:
        return [global_view]

    tile_width = min(width, math.ceil(width / (2 - overlap)))
    tile_height = min(height, math.ceil(height / (2 - overlap)))
    x_positions = [0, width - tile_width]
    y_positions = [0, height - tile_height]
    views = [global_view]
    for top in y_positions:
        for left in x_positions:
            crop = image.crop((left, top, left + tile_width, top + tile_height))
            views.append(ResizeWithPadding(size, channel_mean)(crop))
    return views


example_row = manifest.sample(1, random_state=SEED).iloc[0]
example_views = global_and_local_views(load_rgb_image(example_row.source_path))
fig, axes = plt.subplots(1, len(example_views), figsize=(3 * len(example_views), 3))
axes = np.atleast_1d(axes)
for number, (axis, view) in enumerate(zip(axes, example_views)):
    axis.imshow(view); axis.axis('off'); axis.set_title('global' if number == 0 else f'tile {number}')
plt.tight_layout(); plt.show()


## 9. Quality flags dan ekspor

Flag brightness/contrast tetap disimpan agar eksperimen selanjutnya dapat dibandingkan. Tidak ada pixel yang dikoreksi pada V1.


In [ ]:
manifest['dark_candidate'] = manifest.luminance_mean.lt(0.20)
manifest['bright_candidate'] = manifest.luminance_mean.gt(0.80)
manifest['low_contrast_candidate'] = manifest.luminance_std.lt(0.10)
manifest['narrow_dynamic_range_candidate'] = (manifest.luminance_p99 - manifest.luminance_p01).lt(0.35)
manifest.to_csv(OUTPUT_DIR / 'training_manifest.csv', index=False)

summary = {
    'policy': 'V1 exact deduplication + strict near-duplicate grouping + letterbox RGB',
    'source_images': int(len(metadata)),
    'included_images': int(len(manifest)),
    'included_class_counts': {label: int(manifest.label.eq(label).sum()) for label in CLASSES},
    'excluded_by_reason': {str(k): int(v) for k, v in exclusions.decision_reason.value_counts().items()},
    'exact_cross_label_groups': int(len(conflict_hashes)),
    'near_duplicate_candidate_pairs': int(len(near_pairs)),
    'near_duplicate_cross_label_pairs': int(near_pairs.cross_label.sum()) if len(near_pairs) else 0,
    'split_groups': int(manifest.split_group.nunique()),
    'folds': FOLDS,
    'image_size': IMAGE_SIZE,
    'normalization_mean': channel_mean.tolist(),
    'normalization_std': channel_std.tolist(),
    'quality_flags': {
        key: int(manifest[key].sum()) for key in [
            'dark_candidate', 'bright_candidate', 'low_contrast_candidate', 'narrow_dynamic_range_candidate'
        ]
    },
    'source_files_modified': False,
    'labels_changed': False,
    'photometric_correction_applied': False,
    'external_model_weights_used': False,
}
(OUTPUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))

# Validasi akhir
assert len(manifest) == manifest.pixel_sha256.nunique()
assert manifest.groupby('split_group').fold.nunique().max() == 1
assert not manifest.source_path.map(lambda value: Path(value).is_file()).eq(False).any()

archive = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Output directory:', OUTPUT_DIR)
print('ZIP:', archive)


## 10. Pemakaian untuk training berikutnya

Gunakan `training_manifest.csv` sebagai satu-satunya daftar input training. Kolom penting:

- `source_path`: file sumber yang dibaca dataloader;
- `label`: label folder asli;
- `pixel_sha256`: identitas decoded pixel;
- `split_group`: grup near-duplicate untuk mencegah leakage;
- `fold`: fold validation tetap;
- flag kualitas: metadata untuk error analysis, bukan alasan penghapusan otomatis.

Urutan eksperimen yang disarankan:

1. RGB + letterbox 320 tanpa photometric correction;
2. sampler dengan `strength=0.5` dan bandingkan dengan distribusi alami;
3. global + tile view;
4. brightness/contrast augmentation ringan;
5. conditional gamma/CLAHE sebagai eksperimen terpisah.

Pilih preprocessing berdasarkan rata-rata dan variasi grouped macro-F1 lima fold, bukan hanya public leaderboard.
